# 12 · MLOps — From Notebook to Production Service

A model in a notebook helps no one. **MLOps** is the discipline of turning it into a
reliable, reproducible, monitored service. This notebook walks the reference architecture
you're standardizing on, and shows the parts we *can* run live here.

**The reference stack**

| Concern | Tool |
|---|---|
| Language & serving | Python 3.9+, **FastAPI** |
| Experiment tracking | **MLflow** |
| Data versioning | **DVC** |
| Lint / format | **Ruff** (via pre-commit) |
| Container & CI/CD | **Docker**, **GitHub Actions**, **Make** |

The through-line: the fitted `Pipeline` from notebook 10 is the *artifact* that flows through
training → tracking → serving → monitoring.

## 1. The standard project layout

Every project uses the same shape, so anyone can navigate any repo. Data and code are strictly
separated; data is versioned by DVC (not Git), code lives under `src/`, serving and training
are distinct modules.

```
mlops-standard-template/
├── .dvc/                     # data version control config
├── .github/workflows/ci-cd.yaml
├── .pre-commit-config.yaml   # Ruff runs before every commit
├── Makefile                  # make train / make serve / make lint
├── pyproject.toml            # central config: Ruff, Pytest
├── requirements.txt
├── data/                     # DVC-tracked, Git-ignored
├── logs/
├── deployment/Dockerfile
└── src/
    ├── utils/logger.py       # structured logging
    ├── features/             # feature engineering
    ├── models/
    │   ├── train.py          # MLflow-tracked training
    │   └── evaluate.py
    ├── serving/
    │   ├── app.py            # FastAPI app
    │   └── schemas.py        # Pydantic validation
    └── monitoring/drift_detector.py
```

## 2. Train and produce the artifact (this runs)

We train a real pipeline and save it exactly as `src/models/train.py` would. This `.joblib`
file is the single artifact deployment needs.

In [1]:
import joblib, os, json
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

ROOT = Path("/home/claude/mlops_demo")
(ROOT / "artifacts").mkdir(parents=True, exist_ok=True)

d = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.25,
                                      random_state=42, stratify=d.target)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf",   RandomForestClassifier(n_estimators=200, random_state=42)),
])
pipe.fit(Xtr, ytr)

metrics = {
    "cv_accuracy": float(cross_val_score(pipe, d.data, d.target, cv=5).mean()),
    "test_roc_auc": float(roc_auc_score(yte, pipe.predict_proba(Xte)[:, 1])),
}
joblib.dump(pipe, ROOT / "artifacts" / "model.joblib")
(ROOT / "artifacts" / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("saved artifact + metrics:", metrics)

saved artifact + metrics: {'cv_accuracy': 0.9578481602235678, 'test_roc_auc': 0.9949685534591195}


## 3. Experiment tracking with MLflow

In production, `train.py` logs parameters, metrics, and the artifact to **MLflow** so every run
is reproducible and comparable. If MLflow is installed here we log a real run to a local SQLite
backend; if not, the code below is exactly the pattern you'd use.

```python
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("cancer_detection")

with mlflow.start_run():
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_metric("cv_accuracy", metrics["cv_accuracy"])
    mlflow.log_metric("test_roc_auc", metrics["test_roc_auc"])
    mlflow.sklearn.log_model(pipe, "model")
```

In [2]:
# Run it for real if mlflow is available, else show the pattern gracefully.
try:
    import mlflow, mlflow.sklearn
    mlflow.set_tracking_uri(f"sqlite:///{ROOT/'mlflow.db'}")
    mlflow.set_experiment("cancer_detection")
    with mlflow.start_run():
        mlflow.log_param("model_type", "RandomForest")
        mlflow.log_param("n_estimators", 200)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
    print("Logged a run to MLflow. View it with:  mlflow ui --backend-store-uri sqlite:///mlflow.db")
except ImportError:
    print("mlflow not installed in this sandbox — install with `pip install mlflow`.")
    print("The pattern above is what src/models/train.py uses in the real project.")

mlflow not installed in this sandbox — install with `pip install mlflow`.
The pattern above is what src/models/train.py uses in the real project.


## 4. Structured logging

The reference `logger.py` writes append-only, per-module log files plus console output — so
`training.log` and `webapp.log` stay separate and greppable. A minimal version of that pattern:

In [3]:
import logging, sys
from pathlib import Path

def get_logger(name, log_dir=ROOT / "logs"):
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    if not logger.handlers:                       # avoid duplicate handlers on re-run
        logger.setLevel(logging.DEBUG)
        fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
                                datefmt="%Y-%m-%d %H:%M:%S")
        fh = logging.FileHandler(log_dir / f"{name}.log", mode="a", encoding="utf-8")
        fh.setFormatter(fmt)
        ch = logging.StreamHandler(sys.stdout); ch.setFormatter(fmt); ch.setLevel(logging.INFO)
        logger.addHandler(fh); logger.addHandler(ch); logger.propagate = False
    return logger

log = get_logger("training")
log.info("Training pipeline finished. test_roc_auc=%.3f", metrics["test_roc_auc"])
print("\nlog file written to:", ROOT / "logs" / "training.log")

2026-07-20 18:05:49 | INFO     | training | Training pipeline finished. test_roc_auc=0.995



log file written to: /home/claude/mlops_demo/logs/training.log


## 5. Serving with FastAPI + Pydantic

Serving loads the artifact once at startup and exposes a `/predict` endpoint. **Pydantic**
schemas validate every incoming request, so bad input is rejected with a clear error instead
of crashing the model. Here's the shape (`src/serving/app.py` + `schemas.py`):

```python
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib, numpy as np

class PredictRequest(BaseModel):
    features: list[float] = Field(..., min_length=30, max_length=30)

class PredictResponse(BaseModel):
    prediction: int
    probability: float
    model_version: str

app = FastAPI(title="Cancer Detection API")
model = joblib.load("artifacts/model.joblib")   # load once, reuse

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    x = np.array(req.features).reshape(1, -1)
    proba = float(model.predict_proba(x)[0, 1])
    return PredictResponse(prediction=int(proba > 0.5),
                           probability=proba, model_version="v1.0")
```

We can't run a live server in this notebook, but we can execute the exact prediction logic the
endpoint would run:

In [4]:
import numpy as np
model = joblib.load(ROOT / "artifacts" / "model.joblib")

def predict_endpoint(features):
    # Mirror of the FastAPI /predict handler
    if len(features) != 30:
        return {"error": f"expected 30 features, got {len(features)}"}   # Pydantic would do this
    x = np.array(features, dtype=float).reshape(1, -1)
    proba = float(model.predict_proba(x)[0, 1])
    return {"prediction": int(proba > 0.5), "probability": round(proba, 4), "model_version": "v1.0"}

sample = d.data[0].tolist()
print("valid request  ->", predict_endpoint(sample))
print("invalid request->", predict_endpoint([1.0, 2.0, 3.0]))

valid request  -> {'prediction': 0, 'probability': 0.04, 'model_version': 'v1.0'}
invalid request-> {'error': 'expected 30 features, got 3'}


## 6. Monitoring: catching data drift

A deployed model silently rots when live data drifts away from training data. A simple detector
compares the distribution of each incoming feature to the training distribution (here via a
Kolmogorov–Smirnov test) and flags large shifts for retraining.

In [5]:
from scipy.stats import ks_2samp

def detect_drift(reference, live, alpha=0.01):
    # Flag features whose live distribution differs significantly from training
    drifted = []
    for j in range(reference.shape[1]):
        stat, p = ks_2samp(reference[:, j], live[:, j])
        if p < alpha:
            drifted.append(j)
    return drifted

rng = np.random.RandomState(0)
no_drift = Xte                                   # same distribution as training
drifted  = Xte + rng.normal(3, 1, Xte.shape)     # shifted "live" data

print("drifted features (healthy traffic):", detect_drift(Xtr, no_drift))
print("drifted features (shifted traffic):", len(detect_drift(Xtr, drifted)), "of 30 flagged")

drifted features (healthy traffic): []
drifted features (shifted traffic): 26 of 30 flagged


## 7. The glue: Docker, Make, CI/CD

The pieces that make it reproducible and automated (reference snippets — not run here):

**Dockerfile** — pin the Python version, copy `requirements.txt` first for layer caching, add a
healthcheck:
```dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
HEALTHCHECK --interval=30s CMD curl -f http://localhost:8000/health || exit 1
CMD ["uvicorn", "src.serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Makefile** — one command per task, so nobody memorizes long invocations:
```make
lint:   ; ruff check . && ruff format .
train:  ; python src/models/train.py
serve:  ; uvicorn src.serving.app:app --reload
docker: ; docker build -t mlops-app -f deployment/Dockerfile .
```

**GitHub Actions** — on every push: lint → test → build. Bad code never reaches `main`.
```yaml
on: [push]
jobs:
  ci:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.9" }
      - run: pip install -r requirements.txt
      - run: ruff check .
      - run: pytest -q
```

## Recap — the production loop

1. **Structure** every project the same way (`src/`, DVC data, `deployment/`).
2. **Train** into a single artifact; **track** params/metrics/model with MLflow.
3. **Log** with structured, per-module files.
4. **Serve** with FastAPI + Pydantic validation, loading the artifact once.
5. **Monitor** for drift; drift → retrain → new tracked run.
6. **Automate** with Docker + Make + GitHub Actions so it's reproducible and hands-off.

You've now gone from a NumPy array (notebook 01) to a monitored production service. The
`PROJECTS.md` file lists 50 projects to practice this whole loop end to end.